# **AquaSense AI — AIML Assignment 2: Feature Engineering**



1.Arya Bhat-202401040044  Roll No:10

2.Harshal Mekhe-202401040051  Roll No:14

3.Prerana Kamankar-202401040082  Roll No:24


**Batch:A1**



## **Aim:**
To implement various Feature Engineering techniques on real-world Maharashtra rainfall data by collecting, cleaning, transforming, and analyzing raw rainfall observations to create meaningful features that can be used for water-resource analysis and future rainfall prediction.

## **Problem Statement:**

Raw rainfall data contains daily observations that may include missing values, duplicate records, inconsistent formats, and limited information for direct analysis. It is difficult to identify rainfall patterns, seasonal changes, and rainfall intensity using only raw data.

Therefore, the aim is to apply Feature Engineering techniques to transform raw rainfall data into meaningful numerical and categorical features that improve data understanding and support future water-sustainability analysis.

## **Objectives:**

1. To collect real-world rainfall data from the National Water Data Portal.
2. To understand the structure, attributes, and quality of the collected dataset.
3. To clean the dataset by handling missing values, duplicates, and invalid records.
4. To apply statistical techniques to understand rainfall distribution and patterns.
5. To create meaningful features using date, month, season, and rainfall measurements.
6. To apply lag, rolling average, and seasonal analysis techniques.
7. To perform feature selection using variance, correlation, and mutual information.
8. To encode categorical variables and scale numerical features.
9. To generate a final feature-engineered dataset suitable for future machine learning applications.

## 1. Dataset source and data understanding

**Primary source:** National Water Data Portal (NWDP), Ministry of Jal Shakti / National Water Informatics Centre. The dataset is titled **“Rainfall (Manual - Daily), Maharashtra Surface Water Department.”** The portal describes records as daily manual rainfall measurements from monitoring stations, with date, station/geographical hierarchy and rainfall in millimetres. The selected resource covers **Maharashtra, 2021–2025**.

- Dataset page: https://nwdp.nwic.gov.in/en/dataset/rainfall-maharashtra-surface-water-manual-daily
- Resource preview/schema: https://nwdp.nwic.gov.in/dataset/rainfall-maharashtra-surface-water-manual-daily/resource/f7e3f716-7189-4288-996f-acf9ef139b44/view/ad2bb7aa-4bf1-4a91-91eb-580fcc8a24ea

### Expected raw attributes
The published preview lists fields such as `Station`, `State`, `District`, `Tehsil`, `Block`, `Village`, `River`, `Basin`, `Latitude`, `Longitude`, `Data Acquisition Time`, and `Manual Daily Rainfall (mm)`. The scraper below checks the live page and the notebook prints the exact columns actually downloaded. If the portal changes the resource, inspect the printed columns before proceeding.

### Why this data is relevant
Rainfall amount, rainy-day frequency, heavy-rain events, and seasonal variability are useful indicators for water-resource planning. Feature engineering will convert daily station readings into monthly station-level summaries that are easier to analyze and can later be used by AquaSense forecasting models.

In [ ]:
# 2. Install / import libraries
%pip -q install requests beautifulsoup4 pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import io
import re
import calendar
import warnings
from urllib.parse import urljoin

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
from IPython.display import display
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Web scraping: discover the official data resource

We first request the NWDP HTML dataset page, parse its anchors with BeautifulSoup, locate the 2021–2025 Maharashtra resource, and then inspect its resource page for the CSV download link. This is the web-scraping stage. The official CSV is then downloaded from the link exposed by the portal.

The portal's page structure may change. A known official download endpoint is kept as a fallback so the notebook can still proceed if the download anchor is rendered differently by the website.

In [ ]:
DATASET_PAGE_URL = (
    "https://nwdp.nwic.gov.in/en/dataset/"
    "rainfall-maharashtra-surface-water-manual-daily"
)
RESOURCE_PAGE_URL_FALLBACK = (
    "https://nwdp.nwic.gov.in/en/dataset/"
    "rainfall-maharashtra-surface-water-manual-daily/resource/"
    "f7e3f716-7189-4288-996f-acf9ef139b44"
)
OFFICIAL_CSV_FALLBACK_URL = (
    "https://nwdp.nwic.gov.in/dataset/e562d51d-78f9-4d53-95e3-faaec7bab7de/"
    "resource/f7e3f716-7189-4288-996f-acf9ef139b44/download/"
    "rainfall_manual_daily_maharashtra_sw_mh_2021_2025.csv"
)

HEADERS = {
    "User-Agent": "AquaSenseAI-Student-Project/1.0 (educational data analysis)"
}

session = requests.Session()
session.headers.update(HEADERS)

page_links = []
resource_page_url = RESOURCE_PAGE_URL_FALLBACK
try:
    response = session.get(DATASET_PAGE_URL, timeout=40)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    page_title = soup.title.get_text(" ", strip=True) if soup.title else "NWDP rainfall dataset"
    print("Scraped page title:", page_title)
    print("Page URL:", response.url)

    # Extract visible link text and destination from the dataset listing page.
    for a in soup.find_all("a", href=True):
        label = a.get_text(" ", strip=True)
        href = urljoin(response.url, a["href"])
        if label or "/resource/" in href or "/download/" in href:
            page_links.append({"link_text": label, "url": href})

    links_df = pd.DataFrame(page_links).drop_duplicates()
    display(links_df[links_df["link_text"].str.contains("Rainfall|Download|Preview", case=False, na=False)].head(25))

    target_mask = links_df["link_text"].str.contains(
        r"Rainfall Maharashtra.?SW Maharashtra \(2021\s*[-–]\s*2025\)",
        case=False,
        regex=True,
        na=False,
    )
    resource_candidates = links_df.loc[target_mask, "url"].tolist()
    resource_page_url = next(
        (u for u in resource_candidates if "/resource/" in u),
        RESOURCE_PAGE_URL_FALLBACK,
    )
except Exception as exc:
    print("Could not scrape the listing page in this session:", repr(exc))
    print("Using the official NWDP resource page URL already identified for this assignment.")
    links_df = pd.DataFrame(columns=["link_text", "url"])

print("\nSelected resource page:", resource_page_url)

In [ ]:
# Scrape the selected resource page and search for a downloadable CSV URL.
resource_links = []
download_url = OFFICIAL_CSV_FALLBACK_URL
try:
    resource_response = session.get(resource_page_url, timeout=40)
    resource_response.raise_for_status()
    resource_soup = BeautifulSoup(resource_response.text, "html.parser")

    for a in resource_soup.find_all("a", href=True):
        label = a.get_text(" ", strip=True)
        href = urljoin(resource_response.url, a["href"])
        if ".csv" in href.lower() or "/download/" in href.lower() or "download" in label.lower():
            resource_links.append({"link_text": label, "url": href})

    resource_links_df = pd.DataFrame(resource_links).drop_duplicates()
    if not resource_links_df.empty:
        display(resource_links_df)

    discovered_downloads = (
        resource_links_df.loc[
            resource_links_df["url"].str.contains(r"\.csv|/download/", case=False, regex=True),
            "url",
        ].tolist()
        if not resource_links_df.empty else []
    )
    if discovered_downloads:
        download_url = discovered_downloads[0]
except Exception as exc:
    print("Could not inspect the resource page in this session:", repr(exc))
    print("Using the official CSV endpoint exposed by the portal's Download link as fallback.")

print("Chosen official CSV endpoint:", download_url)

## 3. Download the raw CSV from the scraped source

The next cell downloads the CSV. If the portal is temporarily unavailable or blocks automated requests, the cell provides a manual-upload fallback: download the CSV using the portal's **Download** button and upload it into Colab.

In [ ]:
from google.colab import files

try:
    csv_response = session.get(download_url, timeout=120)
    csv_response.raise_for_status()
    content_type = csv_response.headers.get("Content-Type", "").lower()

    # A CSV response may be labelled text/csv, application/octet-stream or text/plain.
    if "html" in content_type or csv_response.content[:100].lower().lstrip().startswith(b"<!doctype html"):
        raise ValueError(f"The server returned HTML instead of CSV (Content-Type: {content_type}).")

    raw = pd.read_csv(io.BytesIO(csv_response.content), low_memory=False)
    print(f"Downloaded {len(csv_response.content):,} bytes from NWDP.")

except Exception as exc:
    print("Automatic download was unsuccessful:", repr(exc))
    print("Download the 2021–2025 Maharashtra CSV from the NWDP page and upload it below.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No CSV file was uploaded. Please re-run this cell after uploading the official NWDP CSV.")
    uploaded_name = next(iter(uploaded))
    raw = pd.read_csv(io.BytesIO(uploaded[uploaded_name]), low_memory=False)
    print("Loaded manually uploaded file:", uploaded_name)

print("Raw dataset shape:", raw.shape)
print("Raw columns:")
print(raw.columns.tolist())
display(raw.head())

## 4. Understand the raw data before transforming it

Inspect data types, missing values, duplicate rows, date range, and rainfall distribution. We do not automatically replace missing rainfall readings with zero: **missing means not observed, whereas zero means an observed dry day.** This distinction matters for rainfall statistics.

In [ ]:
print("Dataset information:")
raw.info()

missing_summary = (
    raw.isna().sum().to_frame("missing_count")
    .assign(missing_percent=lambda x: 100 * x["missing_count"] / len(raw))
    .sort_values("missing_percent", ascending=False)
)
display(missing_summary.head(25))

print("Exact duplicate rows:", raw.duplicated().sum())

## 5. Standardize and clean important columns

The published resource preview identifies the rainfall measurement and acquisition-time fields. The code below uses flexible column-name matching so minor changes in capitalization or spacing do not break the notebook. If the portal changes its schema substantially, adjust the manual overrides in the next cell.

In [ ]:
def find_column(columns, patterns, required=True):
    """Prefer exact normalized column names, then fall back to partial matches."""
    normalized = {c: re.sub(r"[^a-z0-9]+", "", str(c).lower()) for c in columns}
    normalized_patterns = [re.sub(r"[^a-z0-9]+", "", p.lower()) for p in patterns]

    # First pass: exact match prevents 'District LGD Code' being chosen instead of 'District'.
    for p in normalized_patterns:
        for col, norm in normalized.items():
            if norm == p:
                return col

    # Second pass: partial match handles suffixes such as units in parentheses.
    for p in normalized_patterns:
        for col, norm in normalized.items():
            if p in norm:
                return col

    if required:
        raise KeyError(
            f"Could not find a column matching {patterns}. Available columns: {list(columns)}"
        )
    return None

# Automatic mapping based on the NWDP resource preview.
DATE_COL = find_column(raw.columns, ["Data Acquisition Time", "Observation Date", "Date"])
RAINFALL_COL = find_column(raw.columns, ["Manual Daily Rainfall", "Rainfall"])
STATION_COL = find_column(raw.columns, ["Station"], required=False)
DISTRICT_COL = find_column(raw.columns, ["District"], required=False)
STATE_COL = find_column(raw.columns, ["State"], required=False)
TEHSIL_COL = find_column(raw.columns, ["Tehsil"], required=False)
BLOCK_COL = find_column(raw.columns, ["Block"], required=False)
VILLAGE_COL = find_column(raw.columns, ["Village"], required=False)
LAT_COL = find_column(raw.columns, ["Latitude", "Lat"], required=False)
LON_COL = find_column(raw.columns, ["Longitude", "Long", "Lon"], required=False)

column_map = {
    "date": DATE_COL,
    "rainfall_mm": RAINFALL_COL,
    "station": STATION_COL,
    "district": DISTRICT_COL,
    "state": STATE_COL,
    "tehsil": TEHSIL_COL,
    "block": BLOCK_COL,
    "village": VILLAGE_COL,
    "latitude": LAT_COL,
    "longitude": LON_COL,
}
print("Detected column mapping:")
display(pd.DataFrame({"standard_name": list(column_map.keys()), "source_column": list(column_map.values())}))

In [ ]:
# Build a clean, consistent working table without changing the source file.
clean = pd.DataFrame(index=raw.index)
clean["date"] = pd.to_datetime(raw[DATE_COL], errors="coerce", dayfirst=True)
clean["rainfall_mm"] = pd.to_numeric(
    raw[RAINFALL_COL].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)

for standard_name, source_col in [
    ("station", STATION_COL), ("district", DISTRICT_COL), ("state", STATE_COL),
    ("tehsil", TEHSIL_COL), ("block", BLOCK_COL), ("village", VILLAGE_COL),
]:
    clean[standard_name] = raw[source_col].astype("string").str.strip() if source_col else pd.NA

for standard_name, source_col in [("latitude", LAT_COL), ("longitude", LON_COL)]:
    clean[standard_name] = pd.to_numeric(raw[source_col], errors="coerce") if source_col else np.nan

# A station name may be blank; construct a transparent location key from available geography.
location_parts = [c for c in ["station", "district", "tehsil", "block", "village"] if c in clean.columns]
clean["station_key"] = clean[location_parts].fillna("").astype(str).agg(" | ".join, axis=1)
clean["station_key"] = clean["station_key"].str.strip(" |")
clean.loc[clean["station_key"].eq(""), "station_key"] = "UNKNOWN_STATION"

before_rows = len(clean)
clean = clean.dropna(subset=["date", "rainfall_mm"]).copy()
clean = clean[clean["rainfall_mm"] >= 0].copy()  # Negative rainfall is invalid.
print("Rows removed due to invalid date/rainfall or negative rainfall:", before_rows - len(clean))

# De-duplicate repeated observations for the same station and date by averaging them.
metadata_cols = [c for c in ["district", "state", "tehsil", "block", "village", "latitude", "longitude"] if c in clean.columns]
agg_dict = {"rainfall_mm": "mean"}
agg_dict.update({c: "first" for c in metadata_cols})
daily = (
    clean.groupby(["station_key", "date"], as_index=False)
    .agg(agg_dict)
    .sort_values(["station_key", "date"])
    .reset_index(drop=True)
)

daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month

display(daily.head())
print("Clean daily data shape:", daily.shape)
print("Date range:", daily["date"].min(), "to", daily["date"].max())
print("Unique stations:", daily["station_key"].nunique())
print("Negative rainfall values remaining:", (daily["rainfall_mm"] < 0).sum())

## 6. Statistical summary and initial visual analysis

These summaries satisfy the statistical-analysis part of the assignment and help justify feature construction. Rainfall can be highly skewed because many days have zero or low rainfall while a smaller number of days have heavy rainfall. We therefore inspect mean, median, standard deviation, quantiles and outliers rather than relying on the mean alone.

In [ ]:
print("Rainfall descriptive statistics (mm/day):")
display(daily["rainfall_mm"].describe(percentiles=[.25, .50, .75, .90, .95, .99]).to_frame())

stats_table = pd.DataFrame({
    "Mean": [daily["rainfall_mm"].mean()],
    "Median": [daily["rainfall_mm"].median()],
    "Mode": [daily["rainfall_mm"].mode().iloc[0] if not daily["rainfall_mm"].mode().empty else np.nan],
    "Variance": [daily["rainfall_mm"].var()],
    "Standard deviation": [daily["rainfall_mm"].std()],
    "Skewness": [daily["rainfall_mm"].skew()],
})
display(stats_table.T.rename(columns={0: "Value"}))

q1, q3 = daily["rainfall_mm"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_count = ((daily["rainfall_mm"] < lower_fence) | (daily["rainfall_mm"] > upper_fence)).sum()
print(f"IQR outlier rule: lower={lower_fence:.2f} mm, upper={upper_fence:.2f} mm")
print("Observations beyond IQR fences:", int(outlier_count))
print("Note: intense rainfall observations are not deleted automatically; they may be genuine extreme events.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(daily["rainfall_mm"], bins=50, ax=axes[0])
axes[0].set_title("Distribution of daily rainfall")
axes[0].set_xlabel("Daily rainfall (mm)")
sns.histplot(np.log1p(daily["rainfall_mm"]), bins=50, ax=axes[1])
axes[1].set_title("Log(1 + daily rainfall) distribution")
axes[1].set_xlabel("log(1 + rainfall)")
plt.tight_layout()
plt.show()

monthly_day_summary = daily.groupby("month")["rainfall_mm"].mean().reindex(range(1, 13))
plt.figure(figsize=(11, 4))
sns.barplot(x=monthly_day_summary.index, y=monthly_day_summary.values)
plt.title("Average observed daily rainfall by calendar month")
plt.xlabel("Month number")
plt.ylabel("Mean daily rainfall (mm)")
plt.show()

## 7. Feature Engineering — convert daily readings into station-month features

A raw daily observation is useful, but water-resource analysis often needs monthly and seasonal summaries. We aggregate each station separately, so rainfall observations from different stations are **not added together as though they were one physical rain gauge**.

### Features to be created

| Engineered feature | How it is calculated | Why it is useful |
|---|---|---|
| `rainfall_sum_observed_mm` | Sum of available daily readings in a station-month | Observed monthly rainfall total |
| `mean_daily_rainfall_mm` | Mean of observed daily readings | Typical daily rainfall level |
| `max_daily_rainfall_mm` | Maximum daily reading | Identifies intense rainfall days |
| `rainy_days` | Number of observed days with rainfall greater than 0 | Frequency of rainfall |
| `heavy_rain_days_20mm` | Count of days with rainfall at least 20 mm | Heavy-rain frequency indicator |
| `very_heavy_rain_days_50mm` | Count of days with rainfall at least 50 mm | Very-heavy-rain frequency indicator |
| `rainy_day_fraction` | Rainy days / observed days | Share of observed days with rain |
| `rainfall_intensity_mm_per_rainy_day` | Observed rainfall sum / rainy days | Average amount on rainy days |
| `observation_coverage` | Observed days / calendar days in month | Indicates completeness of a monthly total |
| `month_sin`, `month_cos` | Cyclical encoding of month | Represents December–January continuity |
| `season` | Monsoon / post-monsoon / winter / pre-monsoon | Seasonal context |
| `rainfall_lag_1m` | Previous observed month’s rainfall | Recent history |
| `rainfall_prev3m_mean` | Mean of previous up to three months | Short-term trend |
| `rainfall_anomaly_vs_prior_climatology` | Current month minus past-year same-month average | Deviation from prior seasonal baseline |

**Coverage caution:** `rainfall_sum_observed_mm` is a sum of readings that exist in the source. It is not silently extrapolated to a full month when observations are missing. `observation_coverage` tells us how complete the month is. Filter low-coverage months for any later predictive modelling.

In [ ]:
# Add calendar fields on daily records.
daily["year_month"] = daily["date"].dt.to_period("M").dt.to_timestamp()
daily["month_number"] = daily["date"].dt.month

monthly = (
    daily.groupby(["station_key", "year_month"], as_index=False)
    .agg(
        rainfall_sum_observed_mm=("rainfall_mm", "sum"),
        mean_daily_rainfall_mm=("rainfall_mm", "mean"),
        median_daily_rainfall_mm=("rainfall_mm", "median"),
        max_daily_rainfall_mm=("rainfall_mm", "max"),
        std_daily_rainfall_mm=("rainfall_mm", "std"),
        observed_days=("rainfall_mm", "count"),
        rainy_days=("rainfall_mm", lambda s: int((s > 0).sum())),
        heavy_rain_days_20mm=("rainfall_mm", lambda s: int((s >= 20).sum())),
        very_heavy_rain_days_50mm=("rainfall_mm", lambda s: int((s >= 50).sum())),
        district=("district", "first"),
        state=("state", "first"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
    )
    .sort_values(["station_key", "year_month"])
    .reset_index(drop=True)
)

monthly["year"] = monthly["year_month"].dt.year
monthly["month_number"] = monthly["year_month"].dt.month
monthly["days_in_month"] = monthly["year_month"].dt.days_in_month
monthly["observation_coverage"] = monthly["observed_days"] / monthly["days_in_month"]
monthly["rainy_day_fraction"] = monthly["rainy_days"] / monthly["observed_days"].replace(0, np.nan)
monthly["heavy_rain_day_fraction"] = monthly["heavy_rain_days_20mm"] / monthly["observed_days"].replace(0, np.nan)
monthly["rainfall_intensity_mm_per_rainy_day"] = (
    monthly["rainfall_sum_observed_mm"] / monthly["rainy_days"].replace(0, np.nan)
)

# Meteorological seasons used for this assignment in Maharashtra.
def get_season(month):
    if month in [6, 7, 8, 9]:
        return "Monsoon"
    if month in [10, 11]:
        return "Post-monsoon"
    if month in [12, 1, 2]:
        return "Winter"
    return "Pre-monsoon"

monthly["season"] = monthly["month_number"].map(get_season)
monthly["month_sin"] = np.sin(2 * np.pi * monthly["month_number"] / 12)
monthly["month_cos"] = np.cos(2 * np.pi * monthly["month_number"] / 12)

print("Monthly feature-engineered dataset:", monthly.shape)
display(monthly.head(10))

In [ ]:
# Add lag, rolling and historical seasonal-baseline features.
monthly = monthly.sort_values(["station_key", "year_month"]).reset_index(drop=True)
by_station = monthly.groupby("station_key")["rainfall_sum_observed_mm"]

monthly["rainfall_lag_1m"] = by_station.shift(1)
monthly["rainfall_prev3m_mean"] = by_station.transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)
monthly["rainfall_prev3m_std"] = by_station.transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=2).std()
)

# Baseline uses only earlier years for the same station and calendar month.
monthly["prior_same_month_mean_mm"] = monthly.groupby(
    ["station_key", "month_number"]
)["rainfall_sum_observed_mm"].transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)
monthly["rainfall_anomaly_vs_prior_climatology"] = (
    monthly["rainfall_sum_observed_mm"] - monthly["prior_same_month_mean_mm"]
)

# Optional target for feature-relevance analysis: rainfall in the immediately following calendar month.
monthly["period_index"] = monthly["year"] * 12 + monthly["month_number"]
monthly["next_period_index"] = monthly.groupby("station_key")["period_index"].shift(-1)
monthly["next_month_rainfall_mm"] = monthly.groupby("station_key")["rainfall_sum_observed_mm"].shift(-1)
monthly.loc[
    monthly["next_period_index"] - monthly["period_index"] != 1,
    "next_month_rainfall_mm",
] = np.nan
monthly = monthly.drop(columns=["period_index", "next_period_index"])

print("Additional lag/anomaly features added.")
display(monthly[[
    "station_key", "year_month", "rainfall_sum_observed_mm", "rainfall_lag_1m",
    "rainfall_prev3m_mean", "prior_same_month_mean_mm",
    "rainfall_anomaly_vs_prior_climatology", "next_month_rainfall_mm"
]].head(12))

## 8. Inspect missingness and data quality after feature engineering

Lag features are expected to be missing for the earliest record of each station. The prior-year seasonal baseline is also unavailable until a previous year is observed for that station and calendar month. These are structural missing values, not necessarily data-entry errors.

In [ ]:
monthly_missing = (
    monthly.isna().sum().to_frame("missing_count")
    .assign(missing_percent=lambda x: 100 * x["missing_count"] / len(monthly))
    .sort_values("missing_percent", ascending=False)
)
display(monthly_missing.head(25))

print("Rows with at least 80% daily observation coverage:", int((monthly["observation_coverage"] >= 0.80).sum()))
print("Rows with a valid next-calendar-month target:", int(monthly["next_month_rainfall_mm"].notna().sum()))

## 9. Visualize the engineered features

The charts below show seasonal patterns, monthly rainfall variation, station/district differences and relationships between engineered numerical features. Interpret these as patterns in the portal's monitoring observations, not as direct estimates of total water availability or household consumption.

In [ ]:
# Mean station-month observed rainfall by month number (descriptive across the collected period).
month_profile = monthly.groupby("month_number")["rainfall_sum_observed_mm"].mean().reindex(range(1, 13))
plt.figure(figsize=(11, 4))
sns.lineplot(x=month_profile.index, y=month_profile.values, marker="o")
plt.title("Average observed station-month rainfall by calendar month")
plt.xlabel("Month")
plt.ylabel("Observed rainfall sum (mm)")
plt.xticks(range(1, 13))
plt.show()

# Compare seasons.
plt.figure(figsize=(9, 4))
sns.boxplot(data=monthly, x="season", y="rainfall_sum_observed_mm", showfliers=False)
plt.title("Station-month rainfall distribution by season")
plt.xlabel("Season")
plt.ylabel("Observed rainfall sum (mm)")
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Monthly average over stations at district level. This is an average of station summaries, not a spatially area-weighted rainfall estimate.
if monthly["district"].notna().any():
    district_summary = (
        monthly.groupby("district", dropna=True)["rainfall_sum_observed_mm"]
        .mean().sort_values(ascending=False).head(12)
    )
    plt.figure(figsize=(12, 5))
    plt.barh(district_summary.index.astype(str), district_summary.values)
    plt.title("Districts with highest mean observed station-month rainfall")
    plt.xlabel("Mean station-month rainfall (mm)")
    plt.ylabel("District")
    plt.gca().invert_yaxis()
    plt.show()
else:
    print("District field is unavailable; skipping district plot.")

In [ ]:
correlation_columns = [
    "rainfall_sum_observed_mm", "mean_daily_rainfall_mm", "median_daily_rainfall_mm",
    "max_daily_rainfall_mm", "std_daily_rainfall_mm", "rainy_days",
    "heavy_rain_days_20mm", "very_heavy_rain_days_50mm", "observation_coverage",
    "rainy_day_fraction", "rainfall_intensity_mm_per_rainy_day", "month_sin", "month_cos",
    "rainfall_lag_1m", "rainfall_prev3m_mean", "rainfall_prev3m_std",
    "rainfall_anomaly_vs_prior_climatology", "next_month_rainfall_mm"
]
correlation_data = monthly[correlation_columns].corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_data, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation heatmap of engineered rainfall features")
plt.show()

## 10. Feature selection and justification

Feature engineering creates candidate variables; feature selection keeps useful, non-redundant variables.

We demonstrate three approaches:

1. **Variance filtering:** remove numeric columns with no variation.
2. **Correlation filtering:** flag highly correlated features (absolute correlation above 0.95) to reduce redundancy.
3. **Mutual information:** rank numeric features by their statistical dependency with `next_month_rainfall_mm`.

`next_month_rainfall_mm` is an optional derived target for evaluating feature relevance. It is not a measured municipal consumption value. The mutual-information ranking is an exploratory feature-screening result; in a later predictive modelling assignment, perform feature selection only on the training period to prevent information leakage.

In [ ]:
NUMERIC_CANDIDATES = [
    "rainfall_sum_observed_mm", "mean_daily_rainfall_mm", "median_daily_rainfall_mm",
    "max_daily_rainfall_mm", "std_daily_rainfall_mm", "observed_days", "rainy_days",
    "heavy_rain_days_20mm", "very_heavy_rain_days_50mm", "observation_coverage",
    "rainy_day_fraction", "heavy_rain_day_fraction", "rainfall_intensity_mm_per_rainy_day",
    "month_sin", "month_cos", "rainfall_lag_1m", "rainfall_prev3m_mean",
    "rainfall_prev3m_std", "prior_same_month_mean_mm", "rainfall_anomaly_vs_prior_climatology",
    "latitude", "longitude"
]

available_features = [c for c in NUMERIC_CANDIDATES if c in monthly.columns]
X_numeric = monthly[available_features].replace([np.inf, -np.inf], np.nan)

# Remove all-missing columns, median-impute the remaining values for the purpose of feature screening.
nonempty_features = [c for c in X_numeric.columns if X_numeric[c].notna().any()]
X_numeric = X_numeric[nonempty_features]
imputer_for_screening = SimpleImputer(strategy="median")
X_screen = pd.DataFrame(
    imputer_for_screening.fit_transform(X_numeric),
    columns=X_numeric.columns,
    index=monthly.index,
)

variance_filter = VarianceThreshold(threshold=0.0)
X_variance = variance_filter.fit_transform(X_screen)
variance_kept = X_screen.columns[variance_filter.get_support()].tolist()
print("Numeric features before variance filtering:", len(X_screen.columns))
print("Features remaining after variance filtering:", len(variance_kept))
print("Removed constant features:", sorted(set(X_screen.columns) - set(variance_kept)))

In [ ]:
# Correlation-based redundancy screening (greedy: remove later features in pairs above threshold).
correlation_threshold = 0.95
corr_abs = X_screen[variance_kept].corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
correlation_dropped = [column for column in upper.columns if (upper[column] > correlation_threshold).any()]
correlation_kept = [c for c in variance_kept if c not in correlation_dropped]

print("Correlation threshold:", correlation_threshold)
print("Features flagged as highly correlated:", correlation_dropped)
print("Features retained after correlation screening:", correlation_kept)

In [ ]:
# Mutual-information feature ranking against the following calendar month's rainfall.
mi_data = monthly[correlation_kept + ["next_month_rainfall_mm"]].replace([np.inf, -np.inf], np.nan).dropna(subset=["next_month_rainfall_mm"])

if len(mi_data) >= 10 and len(correlation_kept) > 0:
    X_mi = mi_data[correlation_kept].copy()
    X_mi = X_mi.fillna(X_mi.median(numeric_only=True))
    y_mi = mi_data["next_month_rainfall_mm"]
    mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
    mi_ranking = pd.DataFrame({"feature": correlation_kept, "mutual_information": mi_scores})
    mi_ranking = mi_ranking.sort_values("mutual_information", ascending=False).reset_index(drop=True)
    display(mi_ranking)
    plt.figure(figsize=(10, max(4, 0.35 * len(mi_ranking))))
    plt.barh(mi_ranking["feature"], mi_ranking["mutual_information"])
    plt.title("Mutual-information ranking for next-month rainfall feature screening")
    plt.xlabel("Mutual information")
    plt.gca().invert_yaxis()
    plt.show()
else:
    mi_ranking = pd.DataFrame(columns=["feature", "mutual_information"])
    print("Not enough next-month target rows for reliable mutual-information screening. Use variance/correlation results and document this limitation.")

## 11. Encode categorical information and scale numerical features

The season is categorical, so we one-hot encode it. Numeric measurements are imputed with their training-independent median for this assignment's prepared feature export and standardized using `StandardScaler`.

**For a future predictive model:** fit the imputer, encoder and scaler using only the training split, then apply the fitted transformations to the test split. Do not fit preprocessing on the full dataset before evaluating a model.

In [ ]:
# Start from selected non-redundant numeric features.
selected_numeric = correlation_kept.copy()
X_final_num = monthly[selected_numeric].replace([np.inf, -np.inf], np.nan)

# Median imputation is used only to create a complete feature matrix for this assignment export.
final_imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    final_imputer.fit_transform(X_final_num),
    columns=selected_numeric,
    index=monthly.index,
)

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X_imputed)
X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=[f"scaled_{c}" for c in selected_numeric],
    index=monthly.index,
)

season_encoded = pd.get_dummies(monthly["season"], prefix="season", dtype=int)
model_ready_features = pd.concat(
    [monthly[["station_key", "district", "year_month", "next_month_rainfall_mm"]], X_scaled, season_encoded],
    axis=1,
)

display(model_ready_features.head())
print("Model-ready feature matrix shape:", model_ready_features.shape)

## 12. Final feature dictionary and outputs

The following table describes the features created by the notebook. Keep this dictionary with the final assignment report so the examiner can see what each feature means and why it was engineered.

In [ ]:
feature_dictionary = pd.DataFrame([
    ["rainfall_sum_observed_mm", "Sum of available daily rainfall readings in a station-month", "Monthly water-availability indicator"],
    ["mean_daily_rainfall_mm", "Mean rainfall across observed days", "Typical daily rainfall"],
    ["max_daily_rainfall_mm", "Maximum observed daily rainfall", "Intensity / extreme-event indicator"],
    ["rainy_days", "Count of observed days with rainfall > 0", "Rainfall frequency"],
    ["heavy_rain_days_20mm", "Count of days with rainfall >= 20 mm", "Heavy-rain frequency"],
    ["very_heavy_rain_days_50mm", "Count of days with rainfall >= 50 mm", "Very-heavy-rain frequency"],
    ["rainy_day_fraction", "Rainy days divided by observed days", "Share of wet days"],
    ["rainfall_intensity_mm_per_rainy_day", "Observed rainfall sum divided by rainy days", "Average rain on wet days"],
    ["observation_coverage", "Observed days divided by days in calendar month", "Data reliability indicator"],
    ["season", "Season derived from calendar month", "Seasonal context"],
    ["month_sin / month_cos", "Cyclical transformation of month number", "Preserves cyclic month order"],
    ["rainfall_lag_1m", "Previous month's rainfall sum for the same station", "Recent history"],
    ["rainfall_prev3m_mean", "Mean rainfall over previous up to three monthly records", "Short-term trend"],
    ["rainfall_anomaly_vs_prior_climatology", "Current month minus earlier same-month station average", "Deviation from historical seasonal pattern"],
    ["next_month_rainfall_mm", "Next calendar month's observed rainfall sum", "Optional target for feature relevance screening"],
], columns=["Feature", "Calculation", "Purpose"])
display(feature_dictionary)

In [ ]:
# Export raw, cleaned, feature-engineered, and model-ready outputs.
raw.to_csv("aquasense_nwdp_raw_scraped.csv", index=False)
daily.to_csv("aquasense_rainfall_daily_cleaned.csv", index=False)
monthly.to_csv("aquasense_rainfall_monthly_feature_engineered.csv", index=False)
model_ready_features.to_csv("aquasense_model_ready_features.csv", index=False)
feature_dictionary.to_csv("aquasense_feature_dictionary.csv", index=False)

print("Saved files:")
for filename in [
    "aquasense_nwdp_raw_scraped.csv",
    "aquasense_rainfall_daily_cleaned.csv",
    "aquasense_rainfall_monthly_feature_engineered.csv",
    "aquasense_model_ready_features.csv",
    "aquasense_feature_dictionary.csv",
]:
    print(" -", filename)

In [ ]:
# Download the important outputs from Colab to your computer.
from google.colab import files
files.download("aquasense_rainfall_monthly_feature_engineered.csv")
files.download("aquasense_model_ready_features.csv")
files.download("aquasense_feature_dictionary.csv")

## 13.conclusion-

This assignment demonstrates how raw daily rainfall observations can be converted into informative features for water-sustainability analysis. The website was accessed programmatically, the official Maharashtra rainfall resource was discovered from its dataset page, and the CSV was downloaded and inspected. The cleaning stage standardized dates and rainfall values, removed invalid negative readings, handled duplicate station-date records, and preserved missing rainfall observations as missing rather than incorrectly treating them as zero.

Daily readings were aggregated into station-month records. New features represented rainfall amount, rainfall frequency, heavy-rain frequency, observation coverage, seasonality, cyclical month position, lagged rainfall, rolling averages and deviation from a prior same-month baseline. Variance filtering, correlation screening and mutual-information ranking demonstrated ways to remove constant or redundant variables and prioritize potentially informative features. One-hot encoding and standardization produced a prepared numerical feature matrix for later machine-learning experiments.

**Limitations:**

 The dataset records rainfall at monitoring stations; it does not directly report city household water consumption, reservoir storage, groundwater levels or population. Rainfall aggregates should not be interpreted as municipal water supply. Coverage varies by station-month, and months with low observation coverage should be excluded or treated carefully in future modelling. Mutual-information rankings are exploratory and should be recomputed using only training data during later predictive modelling.

**Future extension for AquaSense AI:**

Combine these engineered rainfall features with verified reservoir-storage, groundwater-level, population and municipal-consumption data. A separate unstructured-document collection can support the project's RAG knowledge assistant; RAG is not required to complete this Feature Engineering assignment.